# Desire lines vs. real bike infrastructure

Hypothesis: the high-affinity OD pairs from `06_desire_lines.ipynb` should overlap with the City of Toronto's actual bike-lane network far more than average pairs do. If true, bike-share ride patterns are a *validation signal* for infrastructure investment — and the top desire lines are a kind of ghost-map of the bike lane network drawn by cyclist behaviour.

Method:
1. Load the City's cycling network (GeoJSON, EPSG:4326), filter to *real* protected infrastructure (bike lanes, cycle tracks, multi-use trails — not sharrows).
2. Project to a metric CRS, buffer bike lanes by 50m, dissolve into one geometry.
3. For each OD pair, compute `fraction_on_bike_lane = (length of A→B straight line inside the bike-lane buffer) / (length of A→B line)`.
4. Correlate with affinity. Test the hypothesis.

Caveat: we use straight-line A→B, not actual ride routes. The signal should still be strong if bike lanes line up with desire lines, because bike lanes tend to run along the direct route between popular OD pairs.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import duckdb
import geopandas as gpd
from shapely.geometry import LineString
import pydeck as pdk

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

METRIC_CRS = 'EPSG:2952'  # NAD83(CSRS) / MTM zone 10, good for Toronto

## 1. Load + filter bike infrastructure

Keep only *real* protected or separated infrastructure. Exclude sharrows / signed routes — those are paint or signage, not protection.

In [2]:
gdf = gpd.read_file(DATA_RAW / 'cycling_network_4326.geojson')
print('Total segments:', len(gdf))

PROTECTED = {
    'Multi-Use Trail', 'Multi-Use Trail - Entrance', 'Multi-Use Trail - Boulevard',
    'Multi-Use Trail - Existing Connector', 'Multi-Use Trail - Connector',
    'Bike Lane', 'Bike Lane - Contraflow', 'Bike Lane - Buffered',
    'Cycle Track', 'Cycle Track - Contraflow', 'Bi-Directional Cycle Track',
}
infra = gdf[gdf['INFRA_HIGHORDER'].isin(PROTECTED)].copy()
print(f'Protected/real infra segments: {len(infra)}')
print(infra['INFRA_HIGHORDER'].value_counts().head(10))

Total segments: 1538
Protected/real infra segments: 1052
INFRA_HIGHORDER
Multi-Use Trail                         352
Bike Lane                               179
Multi-Use Trail - Entrance              177
Cycle Track                             131
Bike Lane - Contraflow                   78
Multi-Use Trail - Boulevard              45
Bi-Directional Cycle Track               31
Multi-Use Trail - Existing Connector     18
Bike Lane - Buffered                     16
Cycle Track - Contraflow                 15
Name: count, dtype: int64


## 2. Project + buffer + dissolve

50m buffer captures "ride is along this bike lane." Wider buffers blur the test; narrower buffers miss slightly off-street approaches.

In [3]:
infra_m = infra.to_crs(METRIC_CRS)
BUFFER_M = 50
buffered = infra_m.buffer(BUFFER_M)
# Dissolve: union everything into a single multi-polygon.
infra_union = buffered.unary_union
print('Total bike-lane area (km^2):', round(infra_union.area / 1e6, 2))
print('Total bike-lane length (km):', round(infra_m.length.sum() / 1000, 1))

/var/folders/91/r7khp0613j32mk0t5mwylvtw0000gn/T/ipykernel_30045/1015997569.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  infra_union = buffered.unary_union


Total bike-lane area (km^2): 59.05
Total bike-lane length (km): 620.1


## 3. Build OD pairs + gravity affinity (same as notebook 06)

In [4]:
stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['sid'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
stations = stations.dropna(subset=['sid']).set_index('sid')[['name', 'lat', 'lon', 'capacity']]

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")
edges = con.execute('''
    SELECT LEAST(Start_Station_Id, End_Station_Id) AS a,
           GREATEST(Start_Station_Id, End_Station_Id) AS b,
           COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL
      AND Start_Station_Id <> End_Station_Id
    GROUP BY 1, 2
''').fetchdf()
edges = edges.join(stations.add_prefix('a_'), on='a', how='inner') \
             .join(stations.add_prefix('b_'), on='b', how='inner')

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    dlat = lat2 - lat1
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

edges['dist_km'] = haversine_km(edges['a_lat'], edges['a_lon'], edges['b_lat'], edges['b_lon']).clip(lower=0.05)

fit = edges[(edges['a_capacity'] > 0) & (edges['b_capacity'] > 0) & (edges['trips'] > 0)]
slope, intercept = np.polyfit(np.log(fit['dist_km']), np.log(fit['trips']) - np.log(fit['a_capacity']) - np.log(fit['b_capacity']), 1)
alpha, K = -slope, np.exp(intercept)
edges['expected'] = K * edges['a_capacity'].astype(float) * edges['b_capacity'].astype(float) / (edges['dist_km'] ** alpha)
edges['affinity'] = edges['trips'] / edges['expected']

cands = edges[(edges['trips'] >= 300) & (edges['a_capacity'] > 0) & (edges['b_capacity'] > 0) & (edges['dist_km'] >= 0.3) & np.isfinite(edges['affinity'])].copy()
print(f'{len(cands)} candidate pairs')

3348 candidate pairs


## 4. Build GeoDataFrame of A→B lines, compute bike-lane coverage per line

In [5]:
# Build straight-line geometry in metric CRS.
# LineString from (a_lon, a_lat) to (b_lon, b_lat) in EPSG:4326, then to EPSG:2952.
lines_wgs = gpd.GeoSeries(
    [LineString([(r.a_lon, r.a_lat), (r.b_lon, r.b_lat)]) for r in cands.itertuples(index=False)],
    crs='EPSG:4326',
)
lines_m = lines_wgs.to_crs(METRIC_CRS)
line_lengths = lines_m.length  # in meters

# Intersection length with the dissolved bike-lane buffer.
print('Computing intersections (this can take ~30s)...')
inside = lines_m.intersection(infra_union)
cands = cands.copy()
cands['line_m'] = line_lengths.values
cands['on_lane_m'] = inside.length.values
cands['lane_cov'] = cands['on_lane_m'] / cands['line_m'].replace(0, np.nan)
print('Done. Lane coverage stats:')
print(cands['lane_cov'].describe())

Computing intersections (this can take ~30s)...


Done. Lane coverage stats:
count    3348.000000
mean        0.404336
std         0.267369
min         0.000000
25%         0.202935
50%         0.344879
75%         0.546153
max         1.000000
Name: lane_cov, dtype: float64


## 5. Does bike-lane coverage predict affinity?

In [6]:
# Bin candidates by lane-coverage quartile; report mean/median affinity.
cands['cov_bin'] = pd.qcut(cands['lane_cov'], q=4, labels=['Q1 (least)', 'Q2', 'Q3', 'Q4 (most)'], duplicates='drop')
print(cands.groupby('cov_bin', observed=True).agg(
    n=('affinity', 'size'),
    median_affinity=('affinity', 'median'),
    mean_affinity=('affinity', 'mean'),
    median_coverage=('lane_cov', 'median'),
).round(3))

# Top-50 desire lines: what's their mean lane coverage vs random 50?
top50 = cands.sort_values('affinity', ascending=False).head(50)
rnd50 = cands.sample(50, random_state=0)
print(f'\nTop-50 desire-line mean lane coverage: {top50["lane_cov"].mean():.1%}')
print(f'Random-50 mean lane coverage:          {rnd50["lane_cov"].mean():.1%}')
print(f'Ratio: {top50["lane_cov"].mean() / rnd50["lane_cov"].mean():.1f}x')

              n  median_affinity  mean_affinity  median_coverage
cov_bin                                                         
Q1 (least)  837            5.266          7.795            0.131
Q2          837            6.321          8.793            0.277
Q3          837            6.473          8.776            0.431
Q4 (most)   837            6.881          9.331            0.768

Top-50 desire-line mean lane coverage: 36.9%
Random-50 mean lane coverage:          41.0%
Ratio: 0.9x


In [7]:
# Correlation (Spearman — robust to the lognormal tail of affinity).
from scipy.stats import spearmanr, pearsonr
s, sp = spearmanr(cands['lane_cov'], np.log(cands['affinity']))
p, pp = pearsonr(cands['lane_cov'], np.log(cands['affinity']))
print(f'Spearman r(lane_cov, log affinity) = {s:.3f}  (p = {sp:.2e})')
print(f'Pearson  r(lane_cov, log affinity) = {p:.3f}  (p = {pp:.2e})')

Spearman r(lane_cov, log affinity) = 0.091  (p = 1.44e-07)
Pearson  r(lane_cov, log affinity) = 0.069  (p = 5.90e-05)


## 6. Validation map — top-50 desire lines, colored by lane coverage

If the hypothesis holds, the top arcs should be visibly overlapping with bike lanes. Overlay bike lanes as green lines.

In [8]:
# Prep bike-lane layer data in 4326.
infra_wgs = infra.to_crs('EPSG:4326')
lane_records = []
for geom, kind in zip(infra_wgs.geometry, infra_wgs['INFRA_HIGHORDER']):
    if geom.geom_type == 'LineString':
        lane_records.append({'path': [list(c) for c in geom.coords], 'kind': kind})
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            lane_records.append({'path': [list(c) for c in line.coords], 'kind': kind})
lane_df = pd.DataFrame(lane_records)
print('Lane path records:', len(lane_df))

lanes_layer = pdk.Layer(
    'PathLayer',
    data=lane_df,
    get_path='path',
    get_color=[30, 140, 70, 170],
    get_width=2.5,
    width_min_pixels=1.5,
    pickable=False,
)

# Top-50 desire arcs, colored by lane coverage (higher coverage = deeper hue).
top50 = cands.sort_values('affinity', ascending=False).head(50).copy()
def cov_color(c):
    c = 0 if not np.isfinite(c) else min(max(c, 0), 1)
    # low coverage = orange-ish, high coverage = deep magenta
    r = int(255 * (1 - c * 0.4))
    g = int(140 * (1 - c) + 30 * c)
    b = int(60 * (1 - c) + 180 * c)
    return [r, g, b, 230]
top50['color'] = top50['lane_cov'].map(cov_color)
top50['width'] = np.clip(np.log1p(top50['trips']) * 1.0, 1.5, 9.5)
top50['pair'] = top50['a_name'].astype(str) + ' - ' + top50['b_name'].astype(str)
top50['aff_r'] = top50['affinity'].round(1)
top50['cov_r'] = (top50['lane_cov'] * 100).round(1)
top50['trips_k'] = (top50['trips'] / 1000).round(2)
top50['dist_km_r'] = top50['dist_km'].round(2)

arc_layer = pdk.Layer(
    'ArcLayer',
    data=top50[['a_lat', 'a_lon', 'b_lat', 'b_lon', 'color', 'width', 'pair', 'aff_r', 'cov_r', 'trips_k', 'dist_km_r']],
    get_source_position=['a_lon', 'a_lat'],
    get_target_position=['b_lon', 'b_lat'],
    get_width='width',
    get_source_color='color',
    get_target_color='color',
    get_height=0.35,
    pickable=True,
    auto_highlight=True,
)

view = pdk.ViewState(latitude=43.660, longitude=-79.395, zoom=12.1, pitch=50, bearing=-15)
deck = pdk.Deck(
    layers=[lanes_layer, arc_layer],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{pair}\n{trips_k}k trips - {dist_km_r} km - affinity {aff_r}x\nlane coverage: {cov_r}%'},
)
out = DATA_PROC / 'infra_validation_map.html'
deck.to_html(str(out), notebook_display=False)
print('Wrote:', out)

Lane path records: 1268


Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/infra_validation_map.html


## 7. Infrastructure gaps — high-affinity pairs with LOW lane coverage

These are routes where cyclists ride a lot, despite no bike lane. Candidate places to invest.

In [9]:
gaps = cands[(cands['affinity'] > cands['affinity'].quantile(0.90)) & (cands['lane_cov'] < 0.1)].copy()
gaps = gaps.sort_values('trips', ascending=False).head(20)
print(f'{len(gaps)} high-affinity pairs with <10% lane coverage (top-10 % of affinity):')
gaps[['a_name', 'b_name', 'trips', 'dist_km', 'affinity', 'lane_cov']].assign(
    dist_km=lambda d: d['dist_km'].round(2),
    affinity=lambda d: d['affinity'].round(1),
    lane_cov=lambda d: (d['lane_cov'] * 100).round(1),
)

15 high-affinity pairs with <10% lane coverage (top-10 % of affinity):


,a_name,b_name,trips,dist_km,affinity,lane_cov
203842,Windsor St / Newcastle St,Grand Avenue Park,3229,0.92,22.6,0.0
129794,Windsor St / Newcastle St,36 Park Lawn Rd,1781,1.29,32.7,0.0
123736,Marilyn Bell Park Tennis Court,Humber Bay Shores Park / Marine Parade Dr,877,3.26,19.1,3.7
49037,Bloor St W / Dundas St W,Roncesvalles Ave / Fermanagh Ave,827,1.14,20.4,8.5
148276,Humber Bay Shores Park East,Marilyn Bell Park Tennis Court,796,2.82,37.7,8.2
63381,Bloor St W / Dundas St W,Garden Ave / Roncesvalles Ave,795,1.44,28.1,9.7
109568,Beaty Ave / Queen St W,Hanna Ave / Snooker St,752,1.67,33.9,0.0
51015,High Park Subway Station,High Park Ave / Dundas St W,724,1.27,36.4,7.9
51152,King St W / Jameson Ave - SMART,Hanna Ave / Snooker St,549,1.32,27.0,0.0
91647,Judson St / Royal York Rd,Sixth St / Lake Shore Blvd W,513,1.73,24.4,6.9


In [10]:
# Focused "gap map": top-50 high-affinity pairs WITH lowest lane coverage.
# These are the strongest candidates for new protected infrastructure.
gap_candidates = cands[
    (cands['trips'] >= 400) &
    (cands['affinity'] >= cands['affinity'].quantile(0.75)) &
    (cands['lane_cov'] < 0.20) &
    (cands['dist_km'] >= 0.5)
].copy()
gap_top = gap_candidates.sort_values('trips', ascending=False).head(50).copy()
print(f'{len(gap_top)} gap pairs on the map')

gap_top['width'] = np.clip(np.log1p(gap_top['trips']) * 1.1, 2.0, 10.0)
gap_top['pair'] = gap_top['a_name'].astype(str) + ' - ' + gap_top['b_name'].astype(str)
gap_top['aff_r'] = gap_top['affinity'].round(1)
gap_top['cov_r'] = (gap_top['lane_cov'] * 100).round(1)
gap_top['trips_k'] = (gap_top['trips'] / 1000).round(2)
gap_top['dist_km_r'] = gap_top['dist_km'].round(2)
# Deep red for gaps - visually demanding attention.
gap_top['color'] = [[220, 35, 50, 235]] * len(gap_top)

gap_arcs = pdk.Layer(
    'ArcLayer',
    data=gap_top[['a_lat', 'a_lon', 'b_lat', 'b_lon', 'color', 'width', 'pair', 'aff_r', 'cov_r', 'trips_k', 'dist_km_r']],
    get_source_position=['a_lon', 'a_lat'],
    get_target_position=['b_lon', 'b_lat'],
    get_width='width',
    get_source_color='color',
    get_target_color='color',
    get_height=0.45,
    pickable=True,
    auto_highlight=True,
)

gap_view = pdk.ViewState(latitude=43.655, longitude=-79.440, zoom=12.4, pitch=50, bearing=-10)
gap_deck = pdk.Deck(
    layers=[lanes_layer, gap_arcs],
    initial_view_state=gap_view,
    map_style='light',
    tooltip={'text': '{pair}\n{trips_k}k trips - {dist_km_r} km\naffinity {aff_r}x  -  lane coverage {cov_r}%'},
)
gap_out = DATA_PROC / 'infra_gap_map.html'
gap_deck.to_html(str(gap_out), notebook_display=False)
print('Wrote:', gap_out)

50 gap pairs on the map


Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/infra_gap_map.html
